# Advanced follow-up: context, auxiliary targets, and what we learned

This notebook is for people who finish early or want to follow up after the tutorial.

The short version of the investigation:

1. A simple hit-only Transformer learns useful separation but leaves a persistent signal population at intermediate score.
2. Making the Transformer bigger or adding fancier pooling doesn't seem to remove that population.
3. A hybrid model that directly receives the BDT variables performs much better.
4. Perturbation studies showed that cluster geometry and angle-like information mattered most.
5. Adding primitive cluster context `(cluster_x, cluster_y, cluster_z, cluster_r)` helped a lot.
6. Adding auxiliary prediction of simple cluster-size variables made the model nearly match the direct-BDT hybrid, without feeding those variables to the classifier.

This notebook implements the final idea: raw hits + primitive context + optional auxiliary targets.

## Summary of the development path

The original tutorial model used raw hit features `(energy, time, x, y)` with a small Transformer block. This gave a reasonable classifier, typically around **AUC ≈ 0.83–0.835** after useful preprocessing.

We then found a persistent signal bump around intermediate scores, roughly `0.35–0.45`. This bump did not disappear with:

- more layers,
- larger hidden dimension,
- soft pooling,
- multi-head attention,
- focal loss,
- more raw hits per cluster.

A BDT-style hybrid model, where the Transformer also received precomputed cluster variables, reached about **AUC ≈ 0.881**. Perturbing those variables showed that `cluster_size_y`, `cluster_size_tot`, `incident_angle`, `cluster_time`, and `cluster_size_x` were important.

The key realization was that some of this information was not present in the original four raw-hit inputs. In particular, `incident_angle` depends on global detector position, while the raw hit `x,y` coordinates are local pixel information. Adding primitive cluster context `(cluster_x, cluster_y, cluster_z, cluster_r)` raised performance to about **AUC ≈ 0.86**.

Finally, auxiliary training pushed performance close to the direct-BDT hybrid. The best simple auxiliary models were:

- raw hits + `cluster_x,y,z,r` + auxiliary `cluster_size_x`: **AUC ≈ 0.881**,
- raw hits + `cluster_x,y,z,r` + auxiliary `cluster_size_x, cluster_size_y`: **AUC ≈ 0.881**,
- top-5 auxiliary targets reduced the low-score signal bump most strongly.

The important conceptual point is that the classifier is **not** handed the engineered variables at inference. The auxiliary loss only shapes the representation during training.

## What is auxiliary targeting doing?

A normal classifier loss gives sparse supervision:

> this cluster is signal, this cluster is BIB.

It does not say *why* a cluster is signal-like or BIB-like. A powerful model may find a decent shortcut and never learn the full geometry.

Auxiliary training adds extra tasks during training:

> classify the cluster, and also predict a few physically meaningful quantities from the same hidden representation.

For example, asking the network to predict `cluster_size_x` forces its representation to encode spatial extent. That representation then helps the classifier. At inference time, we can ignore the auxiliary head.

This is a compromise between two extremes:

- **direct hybrid:** feed engineered variables into the classifier,
- **pure hit-only:** hope the classifier loss discovers everything.

Auxiliary training says:

> learn from raw/primitive inputs, but use extra supervision to make the learned representation physically useful.

In [ ]:
import sys, os, json, math, time, random
from pathlib import Path

import h5py
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import roc_auc_score

from mucoll_helpers.dataloader import load_pixel_data
from mucoll_helpers.tutorial_utils import (
    build_feature_matrix, bib_rejection_at_signal_efficiency,
    plot_score_distributions, plot_confusion_matrix,
    plot_roc_comparison, plot_leaderboard,
    evaluate_classifier, benchmark_bdt,
)

DATA_DIR = Path('/global/cfs/cdirs/ntrain6/mucoll')

## Configuration

In [ ]:
SIGNAL_H5 = str(DATA_DIR / 'signal.h5')
BIB_H5    = str(DATA_DIR / 'BIB.h5')

MAX_RAW_HITS = 50
TEST_FRACTION = 0.2
VAL_FRACTION  = 0.1
RANDOM_SEED   = 42
BATCH_SIZE    = 256

OUTPUT_DIR = 'advanced_context_aux_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BDT_MODEL_PATH = DATA_DIR / 'bdt_baseline_outputs' / 'bdt_gradient_boosting.joblib'

CONTEXT_KEYS = ['cluster_x', 'cluster_y', 'cluster_z', 'cluster_r']
TOP_AUX_KEYS = ['cluster_size_x', 'cluster_size_y']
# Other useful candidates:
# ['cluster_size_y', 'cluster_size_tot', 'incident_angle', 'cluster_time', 'cluster_size_x']

for required_path in [SIGNAL_H5, BIB_H5]:
    if not Path(required_path).exists():
        raise FileNotFoundError(required_path)


In [ ]:
def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        try:
            torch.zeros(1).cuda()
            device = 'cuda'
        except RuntimeError:
            print('CUDA device found but not usable. Falling back to CPU.')
    return device

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)
device = get_device()
print('Using device:', device)

## Fast loading: use the original dataloader for raw hits

The variable-length HDF5 raw-hit arrays are slow to re-read. We use the existing `load_pixel_data(...)` path for the padded hit tensors, and only read small fixed-size cluster arrays from HDF5.

In [ ]:
data = load_pixel_data(
    SIGNAL_H5, BIB_H5,
    max_hits=MAX_RAW_HITS,
    batch=BATCH_SIZE,
    test_frac=TEST_FRACTION,
    val_frac=VAL_FRACTION,
    seed=RANDOM_SEED,
)

train_loader_base = data['train']
val_loader_base   = data['val']
test_loader_base  = data['test']
labels_all        = np.asarray(data['labels']).astype(int)
idx_train         = np.asarray(data['idx_train'])
idx_val           = np.asarray(data['idx_val'])
idx_test          = np.asarray(data['idx_test'])

In [ ]:
# Robust extraction of X,y tensors from the existing loaders.
# This avoids relying on private dataset attributes.

def unpack_xy_batch(batch):
    if isinstance(batch, dict):
        return batch['X'], batch['y']
    if isinstance(batch, (tuple, list)):
        return batch[0], batch[-1]
    raise TypeError(f'Unknown batch type: {type(batch)}')


def extract_split_tensors(loader, split_name, batch_size=4096):
    ordered_loader = DataLoader(loader.dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    xs, ys = [], []
    for batch in ordered_loader:
        x, y = unpack_xy_batch(batch)
        xs.append(x.detach().cpu())
        ys.append(y.detach().cpu())
    X = torch.cat(xs, dim=0)
    y = torch.cat(ys, dim=0).long()
    print(f'{split_name}: X={tuple(X.shape)}, y={tuple(y.shape)}')
    return X, y

X_train, y_train = extract_split_tensors(train_loader_base, 'train')
X_val, y_val     = extract_split_tensors(val_loader_base, 'val')
X_test, y_test   = extract_split_tensors(test_loader_base, 'test')

## Load fixed-size context and auxiliary targets

In [ ]:
def load_fixed_features(signal_h5, bib_h5, keys):
    arrays = []
    for path in [signal_h5, bib_h5]:
        with h5py.File(path, 'r') as f:
            grp = f['clusters']
            arr = np.stack([grp[k][:] for k in keys], axis=1).astype('float32')
            arrays.append(arr)
    return np.concatenate(arrays, axis=0)

context_all = load_fixed_features(SIGNAL_H5, BIB_H5, CONTEXT_KEYS)
aux_all = load_fixed_features(SIGNAL_H5, BIB_H5, TOP_AUX_KEYS)

# Standardize context and auxiliary targets using train split only.
context_mean = context_all[idx_train].mean(axis=0, keepdims=True)
context_std  = context_all[idx_train].std(axis=0, keepdims=True) + 1e-6
context_all_std = (context_all - context_mean) / context_std

aux_mean = aux_all[idx_train].mean(axis=0, keepdims=True)
aux_std  = aux_all[idx_train].std(axis=0, keepdims=True) + 1e-6
aux_all_std = (aux_all - aux_mean) / aux_std

C_train = torch.tensor(context_all_std[idx_train], dtype=torch.float32)
C_val   = torch.tensor(context_all_std[idx_val], dtype=torch.float32)
C_test  = torch.tensor(context_all_std[idx_test], dtype=torch.float32)

A_train = torch.tensor(aux_all_std[idx_train], dtype=torch.float32)
A_val   = torch.tensor(aux_all_std[idx_val], dtype=torch.float32)
A_test  = torch.tensor(aux_all_std[idx_test], dtype=torch.float32)

assert len(X_train) == len(C_train) == len(A_train)
assert len(X_val) == len(C_val) == len(A_val)
assert len(X_test) == len(C_test) == len(A_test)

train_loader = DataLoader(TensorDataset(X_train, C_train, A_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val, C_val, A_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test, C_test, A_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

## Model: simple hit Transformer + primitive context + optional auxiliary head

In [ ]:
class HitPreprocess(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        x = x.clone()
        # energy >= 0, compress large energy tail
        x[:, :, 0] = torch.log1p(torch.clamp(x[:, :, 0], min=0))
        # preserve sign while compressing huge time tail
        x[:, :, 1] = torch.asinh(x[:, :, 1])
        return x


class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.scale = dim ** -0.5

    def forward(self, x, mask):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        return (attn @ v) * mask


class TransformerBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.att = Attention(dim)
        self.proj1 = nn.Linear(dim, dim)
        self.proj2 = nn.Linear(dim, dim)
        self.activation = nn.GELU()
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x, mask):
        x = x + self.att(self.norm1(x), mask)
        x = self.activation(self.proj1(x)) * mask
        x = x + self.proj2(self.norm2(x)) * mask
        return x


class ContextAuxPixelTransformer(nn.Module):
    def __init__(self, input_dim=4, context_dim=4, hidden_dim=64, num_layers=3, num_classes=2, aux_dim=2):
        super().__init__()
        self.preprocess = HitPreprocess()
        self.input_layer = nn.Linear(input_dim + context_dim, hidden_dim)
        self.blocks = nn.ModuleList([TransformerBlock(hidden_dim) for _ in range(num_layers)])

        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden_dim + context_dim),
            nn.Linear(hidden_dim + context_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_classes),
        )
        self.aux_head = nn.Sequential(
            nn.LayerNorm(hidden_dim + context_dim),
            nn.Linear(hidden_dim + context_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, aux_dim),
        ) if aux_dim > 0 else None

    def forward(self, hits, context):
        mask = (hits[:, :, 0] != 0).unsqueeze(-1).float()
        x = self.preprocess(hits)

        # Broadcast primitive context to every hit, so attention can use detector position.
        context_per_hit = context.unsqueeze(1).expand(-1, x.shape[1], -1)
        x = torch.cat([x, context_per_hit], dim=-1)

        x = self.input_layer(x) * mask
        for block in self.blocks:
            x = block(x, mask)

        pooled = (x * mask).sum(dim=1) / (mask.shape[1] ** 0.5)

        # Also give context directly to the final head.
        rep = torch.cat([pooled, context], dim=-1)
        logits = self.classifier(rep)
        aux_pred = self.aux_head(rep) if self.aux_head is not None else None
        return logits, aux_pred

## Training/evaluation utilities with safe output

In [ ]:
def append_csv(row, path):
    path = Path(path)
    df = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        df = pd.concat([old, df], ignore_index=True)
    df.to_csv(path, index=False)


def evaluate_context_model(model, loader):
    model.eval()
    scores, labels = [], []
    aux_preds, aux_true = [], []
    with torch.no_grad():
        for hits, context, aux, y in loader:
            hits = hits.to(device)
            context = context.to(device)
            logits, aux_pred = model(hits, context)
            prob = torch.softmax(logits, dim=1)[:, 1]
            scores.append(prob.cpu())
            labels.append(y.cpu())
            if aux_pred is not None:
                aux_preds.append(aux_pred.cpu())
                aux_true.append(aux.cpu())
    scores = torch.cat(scores).numpy()
    labels = torch.cat(labels).numpy()
    out = {
        'auc': roc_auc_score(labels, scores),
        'bib_rej_90': bib_rejection_at_signal_efficiency(labels, scores, 0.90),
    }
    if aux_preds:
        pred = torch.cat(aux_preds).numpy()
        true = torch.cat(aux_true).numpy()
        rmse = np.sqrt(np.mean((pred - true) ** 2, axis=0))
        for k, v in zip(TOP_AUX_KEYS, rmse):
            out[f'aux_rmse_{k}'] = float(v)
    return out, labels, scores


def train_context_aux_model(name, aux_weight=0.10, hidden_dim=64, num_layers=3, lr=3e-4, num_epochs=80, patience=10):
    set_seed(RANDOM_SEED)
    model = ContextAuxPixelTransformer(
        input_dim=4,
        context_dim=len(CONTEXT_KEYS),
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        aux_dim=len(TOP_AUX_KEYS),
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    best_auc = -np.inf
    best_row = None
    bad = 0
    t0 = time.time()
    exp_dir = OUTPUT_DIR / name
    exp_dir.mkdir(parents=True, exist_ok=True)

    for epoch in range(1, num_epochs + 1):
        model.train()
        losses = []
        for hits, context, aux, y in train_loader:
            hits = hits.to(device)
            context = context.to(device)
            aux = aux.to(device)
            y = y.to(device)

            opt.zero_grad(set_to_none=True)
            logits, aux_pred = model(hits, context)
            loss = ce(logits, y) + aux_weight * mse(aux_pred, aux)
            loss.backward()
            opt.step()
            losses.append(loss.item())

        val_metrics, _, _ = evaluate_context_model(model, val_loader)
        row = {
            'name': name,
            'epoch': epoch,
            'train_loss': float(np.mean(losses)),
            'val_auc': val_metrics['auc'],
            'val_bib_rej_90': val_metrics['bib_rej_90'],
            'aux_weight': aux_weight,
            'hidden_dim': hidden_dim,
            'num_layers': num_layers,
            'elapsed_s': time.time() - t0,
        }
        row.update({k: v for k, v in val_metrics.items() if k.startswith('aux_rmse_')})
        append_csv(row, OUTPUT_DIR / 'all_epoch_history.csv')

        print(f"Epoch {epoch}: loss={row['train_loss']:.4f}, val_auc={row['val_auc']:.5f}")

        if val_metrics['auc'] > best_auc:
            best_auc = val_metrics['auc']
            best_row = row.copy()
            torch.save(model.state_dict(), exp_dir / 'best_model.pt')
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                print('Early stopping')
                break

    model.load_state_dict(torch.load(exp_dir / 'best_model.pt', map_location=device))
    test_metrics, test_labels, test_scores = evaluate_context_model(model, test_loader)
    best_row.update({
        'best_epoch': best_row['epoch'],
        'test_auc': test_metrics['auc'],
        'test_bib_rej_90': test_metrics['bib_rej_90'],
    })
    append_csv(best_row, OUTPUT_DIR / 'completed_experiments.csv')
    return model, best_row, test_labels, test_scores


## Run one strong advanced model and compare visually


In [ ]:
RUN_TRAINING = True

bdt = None
if BDT_MODEL_PATH.exists():
    print('Loading BDT reference:', BDT_MODEL_PATH)
    bdt = joblib.load(BDT_MODEL_PATH)
else:
    print('BDT reference not found. Run 01_bdt_baseline.ipynb to add it to the comparison plots.')

if RUN_TRAINING:
    model, row, test_labels, test_scores = train_context_aux_model(
        name='context_xyzr_aux_size_xy',
        aux_weight=0.10,
        hidden_dim=64,
        num_layers=3,
        lr=3e-4,
        num_epochs=100,
        patience=15,
    )

    print(f"Best epoch: {row['best_epoch']}  test AUC: {row['test_auc']:.5f}  BIB rejection @ 90% signal: {row['test_bib_rej_90']:.4f}")

    roc_payload = {}
    leaderboard_entries = []

    advanced_results = evaluate_classifier(test_labels, test_scores, 'Context + auxiliary Transformer')
    roc_payload['Context + auxiliary Transformer'] = advanced_results
    plot_score_distributions(test_labels, test_scores, title='Context + auxiliary Transformer score distribution')
    plot_confusion_matrix(test_labels, test_scores, cut=0.5, title='Context + auxiliary Transformer confusion matrix, cut = 0.5')

    advanced_size = sum(p.numel() for p in model.parameters()) * 4
    advanced_fom = advanced_results['auc'] / np.log10(max(advanced_size, 1))
    leaderboard_entries.append(('Context + auxiliary Transformer', advanced_results['auc'], advanced_size, advanced_fom))

    if bdt is not None:
        X_bdt_all = build_feature_matrix(data, data['feature_keys'])
        y_bdt_test = labels_all[idx_test]
        bdt_scores = bdt.predict_proba(X_bdt_all[idx_test])[:, 1]
        bdt_results = evaluate_classifier(y_bdt_test, bdt_scores, 'BDT reference')
        roc_payload['BDT reference'] = bdt_results
        plot_score_distributions(y_bdt_test, bdt_scores, title='BDT reference score distribution')
        plot_confusion_matrix(y_bdt_test, bdt_scores, cut=0.5, title='BDT reference confusion matrix, cut = 0.5')
        bdt_bench = benchmark_bdt(bdt, bdt_results['auc'])
        leaderboard_entries.append(('BDT reference', bdt_bench['auc'], bdt_bench['size_bytes'], bdt_bench['fom']))

    plot_roc_comparison(roc_payload)
    plot_leaderboard(leaderboard_entries)
else:
    print('Set RUN_TRAINING=True to train the advanced model.')


## Suggested follow-up tests

Good compact ablations:

1. no auxiliary loss, same context,
2. auxiliary `cluster_size_x` only,
3. auxiliary `cluster_size_x + cluster_size_y`,
4. top-5 auxiliary targets with a larger weight, if the goal is to reduce the low-score signal bump.

The main result from the study was that `cluster_size_x` and `cluster_size_x + cluster_size_y` were enough to get very close to the direct-BDT hybrid AUC, while the top-5 auxiliary model tended to push the signal score distribution upward the most.

## Can you push the model above an AUC of 0.9?